In [ ]:
from pyspark.sql import functions as F

from config import bronze_table, checkpoint_dir, gold_table

In [ ]:
reviews_stream = spark.readStream.table(bronze_table("review"))

windowed_reviews = (
    reviews_stream
    .withWatermark("date", "10 minutes")
    .groupBy(
        F.window("date", "5 minutes"),
        "business_id",
    )
    .agg(
        F.count("*").alias("review_count"),
        F.avg("stars").alias("avg_stars"),
    )
    .select(
        "business_id",
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "review_count",
        "avg_stars",
    )
)

In [ ]:
query = (
    windowed_reviews.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_dir("review_activity_by_window"))
    .trigger(availableNow=True)
    .toTable(gold_table("review_activity_by_window"))
)

query.awaitTermination()